# Clasificador de Contenido Técnico - Pipeline de NLP y Machine Learning (Producción)
Este notebook muestra el paso a paso del pipeline avanzado desarrollado para limpiar y clasificar textos técnicos en las 5 categorías definidas.

## Paso 1: Importar librerías

In [1]:
import os
import sys
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Agregar la ruta actual al path para importar el text_cleaner local
sys.path.append(os.getcwd())
from text_cleaner import TextCleaner

## Paso 2: Cargar el Dataset y Controlar Nulos

In [2]:
dataset_path = os.path.join("data", "dataset.csv")
df = pd.read_csv(dataset_path)
print(f"Registros iniciales cargados: {df.shape[0]}")
df = df.dropna(subset=['text', 'label'])
print(f"Registros válidos tras quitar nulos: {df.shape[0]}")
print("Distribución de categorías en mi dataset:")
print(df['label'].value_counts())

Registros iniciales cargados: 50
Registros válidos tras quitar nulos: 50
Distribución de categorías en mi dataset:
label
Frontend                   10
Backend                    10
Inteligencia artificial    10
Data Science               10
Ciberseguridad             10
Name: count, dtype: int64


## Paso 3: Limpieza Avanzada de Texto (NLP)
Remueve HTML, URLs, direcciones de correo, caracteres especiales y stop words.

In [3]:
cleaner = TextCleaner()
df['cleaned_text'] = df['text'].apply(cleaner.clean)
df = df[df['cleaned_text'].str.strip() != '']
print(f"Registros listos para entrenar tras remoción de textos vacíos: {df.shape[0]}")

print("--- Ejemplo de Limpieza Avanzada ---")
print(f"Original:\n{df['text'].iloc[0]}\n")
print(f"Limpio:\n{df['cleaned_text'].iloc[0]}")

Registros listos para entrenar tras remoción de textos vacíos: 50
--- Ejemplo de Limpieza Avanzada ---
Original:
Desarrollo de interfaces de usuario interactivas utilizando React, HTML5 y CSS3 moderno con diseño responsivo. Más info en https://react.dev o escribe a contact@frontend.com. <div>React rules!</div>

Limpio:
desarrollo interfaces usuario interactivas utilizando react html css moderno diseño responsivo más info escribe react rules


## Paso 4: División en Entrenamiento y Prueba (Train/Test Split)

In [4]:
X = df['cleaned_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Tamaño de mi set de entrenamiento: {len(X_train)}")
print(f"Tamaño de mi set de prueba: {len(X_test)}")

Tamaño de mi set de entrenamiento: 40
Tamaño de mi set de prueba: 10


## Paso 5: Vectorización TF-IDF

In [5]:
vectorizer = TfidfVectorizer(sublinear_tf=True, min_df=1, norm='l2', encoding='utf-8', ngram_range=(1, 1))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print(f"Número de palabras en mi vocabulario: {X_train_tfidf.shape[1]}")

Número de palabras en mi vocabulario: 361


## Paso 6: Entrenamiento del Modelo (Regresión Logística)

In [6]:
model = LogisticRegression(C=10.0, class_weight='balanced', random_state=42)
model.fit(X_train_tfidf, y_train)
print("¡Modelo entrenado exitosamente!")

¡Modelo entrenado exitosamente!


## Paso 7: Evaluación (Métricas de Rendimiento)

In [7]:
y_pred = model.predict(X_test_tfidf)
print("--- Mi Matriz de Confusión ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Mi Reporte de Clasificación ---")
print(classification_report(y_test, y_pred))

--- Mi Matriz de Confusión ---
[[0 1 0 1 0]
 [1 0 0 0 1]
 [0 0 1 0 1]
 [0 0 0 2 0]
 [1 0 0 0 1]]

--- Mi Reporte de Clasificación ---
                         precision    recall  f1-score   support

                Backend       0.00      0.00      0.00         2
         Ciberseguridad       0.00      0.00      0.00         2
           Data Science       1.00      0.50      0.67         2
               Frontend       0.67      1.00      0.80         2
Inteligencia artificial       0.33      0.50      0.40         2

               accuracy                           0.40        10
              macro avg       0.40      0.40      0.37        10
           weighted avg       0.40      0.40      0.37        10


## Paso 8: Exportación y Serialización

In [8]:
output_dir = os.path.join("..", "microservicio", "models")
os.makedirs(output_dir, exist_ok=True)

model_path = os.path.join(output_dir, "model.joblib")
vectorizer_path = os.path.join(output_dir, "vectorizer.joblib")

joblib.dump(model, model_path)
joblib.dump(vectorizer, vectorizer_path)
print(f"Modelo guardado en: {model_path}")
print(f"Vectorizador guardado en: {vectorizer_path}")

Modelo guardado en: ../microservicio/models/model.joblib
Vectorizador guardado en: ../microservicio/models/vectorizer.joblib
